# 03 · Compact Reformat, Baselines, and Base-Model Selection


### Why reformatting comes first — a measured problem



### A statistical issue you must not miss

Corpus B samples 1,500 instances from **340 resources** — your test split has 150 instances
drawn from just **45 unique resources**. Instances are therefore *not independent*.

A naive bootstrap treats each instance as independent and yields CIs that are too narrow.
`bootstrap_ci` in `mcasm.eval.metrics` takes a `clusters` argument and resamples **resources**,
not instances. **Use it.** Reporting a naive CI here is a real statistical error.


## 1 · Setup


In [1]:
!pip -q install "pydantic>=2.0" scipy 2>&1 | tail -2
from pathlib import Path
import sys, zipfile, json, collections
import numpy as np, pandas as pd

BASE = Path('/content/mcasm_project'); BASE.mkdir(parents=True, exist_ok=True)
for z in ['/content/mcasm.zip', '/content/mcasm_corpus_b.zip']:
    p = Path(z)
    print(f'{p.name:28} {"OK" if p.exists() else "MISSING"}')
    if p.exists():
        with zipfile.ZipFile(p) as f: f.extractall(BASE if 'corpus' not in z else BASE/'corpus_b')

sys.path.insert(0, str(BASE))
for m in [k for k in sys.modules if k.startswith('mcasm')]: del sys.modules[m]
import mcasm; print('mcasm version:', mcasm.__version__, '(need 1.2.0)')

CB = BASE/'corpus_b'; OUT = BASE/'train_ready'; FIG = BASE/'figures'
OUT.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)

mcasm.zip                    OK
mcasm_corpus_b.zip           OK
mcasm version: 1.2.0 (need 1.2.0)


In [2]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'
vram = torch.cuda.get_device_properties(0).total_memory/1024**3 if torch.cuda.is_available() else 0
print(f'GPU: {gpu}  |  VRAM: {vram:.1f} GiB')

if vram >= 38:   TIER, CANDIDATES = 'A100', ['Qwen/Qwen2.5-7B-Instruct','microsoft/Phi-4-mini-instruct','meta-llama/Llama-3.1-8B-Instruct']
elif vram >= 20: TIER, CANDIDATES = 'L4/A10', ['Qwen/Qwen2.5-7B-Instruct','microsoft/Phi-4-mini-instruct']
elif vram >= 14: TIER, CANDIDATES = 'T4/V100', ['Qwen/Qwen2.5-3B-Instruct','microsoft/Phi-4-mini-instruct']
else:            TIER, CANDIDATES = 'CPU', []
print(f'tier: {TIER}\ncandidates: {CANDIDATES}')

GPU: Tesla T4  |  VRAM: 14.6 GiB
tier: T4/V100
candidates: ['Qwen/Qwen2.5-3B-Instruct', 'microsoft/Phi-4-mini-instruct']


## 2 · Reformat to compact prompts


In [3]:
from mcasm.corpus.serialise import to_compact_record, encode_set, parse_model_output

splits = {}
for sp in ['train','validation','test']:
    recs = [json.loads(l) for l in open(CB/f'{sp}.jsonl')]
    splits[sp] = [to_compact_record(r, top_k_justifications=3) for r in recs]
    print(f'{sp:11} {len(splits[sp]):5} instances')

def tot_chars(r): return len(r['messages'][1]['content'])+len(r['messages'][2]['content'])
before = [len(json.loads(l)['messages'][1]['content'])+len(json.loads(l)['messages'][2]['content'])
          for l in open(CB/'test.jsonl')]
after  = [tot_chars(r) for r in splits['test']]
print(f'\np95 chars  before {np.percentile(before,95):.0f}  ->  after {np.percentile(after,95):.0f}'
      f'   ({100*(1-np.mean(after)/np.mean(before)):.0f}% reduction)')
print('\nSAMPLE PROMPT\n' + '-'*70)
print(splits['test'][0]['messages'][1]['content'][:700])

train        1200 instances
validation    150 instances
test          150 instances

p95 chars  before 19772  ->  after 2602   (86% reduction)

SAMPLE PROMPT
----------------------------------------------------------------------
Rank these 13 findings, most urgent first.
Reply with JSON: {"ranking":[ids],"ssvc":{id:class},"why":{id:reason}}
ssvc class is one of Track, Track*, Attend, Act. Give `why` for the top 3 only, one short sentence each, citing only indicators shown below.

res-8725bcaa929f|aws|security|aws_security_group :: public, ports=22,3306,3389,5432, shared_infra
res-4995388362e6|azure|automation|azurerm_automation_account :: links=1
res-058deed59045|aws|s3|aws_s3_bucket :: links=1
res-96c3036c47fd|aws|module|module :: links=2
res-a5a3b5331d23|aws|module|module :: links=2
res-a0d50b9b1416|aws|iam|aws_iam_policy :: priv=admin, wildcard_action
res-a58b0b21b879|aws|module|module :: no risk indicators
res-0


In [4]:
try:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained(CANDIDATES[0] if CANDIDATES else 'Qwen/Qwen2.5-3B-Instruct')
    L = np.array([len(tok(r['messages'][1]['content']+r['messages'][2]['content'])['input_ids'])
                  for r in splits['train'][:400]])
    p99 = np.percentile(L,99)
    MAX_SEQ = int(2**np.ceil(np.log2(p99)))
    print(f'tokens: mean {L.mean():.0f}  p50 {np.percentile(L,50):.0f}  p95 {np.percentile(L,95):.0f}  p99 {p99:.0f}')
    print(f'--> MAX_SEQ_LENGTH = {MAX_SEQ}')
except Exception as e:
    MAX_SEQ = 1024
    print('tokenizer unavailable:', type(e).__name__, '-> defaulting MAX_SEQ =', MAX_SEQ)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

tokens: mean 895  p50 884  p95 1123  p99 1151
--> MAX_SEQ_LENGTH = 2048


## 3 · Baselines B0–B3



In [5]:
from mcasm.schema.ufs import UnifiedFinding
from mcasm.rank.baselines import BASELINES
from mcasm.eval.metrics import evaluate_instance, bootstrap_ci, permutation_test

pool = {}
for line in open(CB/'resource_pool.jsonl'):
    f = UnifiedFinding.model_validate(json.loads(line)); pool[f.finding_id] = f

def run_baseline(name, fn, records):
    rows, clusters = [], []
    for r in records:
        ids = [f['id'] for f in r['findings']]
        objs = [pool[i] for i in ids if i in pool]
        if len(objs) < 4: continue
        pred = fn(objs)
        gold = r['target']['ranking']
        gs = {fid: len(gold)-i for i, fid in enumerate(gold)}
        m = evaluate_instance(pred, gold, gold_scores=gs)
        m['instance_id'] = r['instance_id']
        rows.append(m); clusters.append(gold[0])   # cluster on top-ranked resource
    return pd.DataFrame(rows), clusters

results = {}
for name, fn in BASELINES.items():
    d, cl = run_baseline(name, fn, splits['test'])
    ci = bootstrap_ci(d['kendall_tau_b'].values, cl, n_boot=4000)
    results[name] = {
        'tau_b': ci['point'], 'ci_lo': ci['lo'], 'ci_hi': ci['hi'],
        'spearman': d['spearman_rho'].mean(),
        'ndcg@5': d.get('ndcg@5', pd.Series([np.nan])).mean(),
        'p@3': d.get('precision@3', pd.Series([np.nan])).mean(),
        'n_units': ci['n_units'],
    }
    results[name+'_raw'] = d

tbl = pd.DataFrame({k: v for k, v in results.items() if not k.endswith('_raw')}).T
display(tbl.round(4))

,tau_b,ci_lo,ci_hi,spearman,ndcg@5,p@3,n_units
B0_random,-0.0143,-0.0409,0.0203,-0.0238,0.6140,0.2333,13.0
B1_scanner_severity,-0.0052,-0.0775,0.0500,-0.0041,0.6377,0.2822,13.0
B2_cvss_epss,-0.0052,-0.0775,0.0500,-0.0041,0.6377,0.2822,13.0
B3_composite,0.9408,0.9190,0.9694,0.9724,0.9815,0.8733,13.0


In [6]:
# Demonstrate the clustering effect -- put this figure in the thesis
d = results['B1_scanner_severity_raw']['kendall_tau_b'].values
cl = [r['target']['ranking'][0] for r in splits['test']][:len(d)]
naive = bootstrap_ci(d, None, n_boot=4000)
clust = bootstrap_ci(d, cl,  n_boot=4000)
display(pd.DataFrame([
    {'bootstrap':'naive (by instance)','n_units':naive['n_units'],'lo':naive['lo'],'hi':naive['hi'],'width':naive['hi']-naive['lo']},
    {'bootstrap':'clustered (by resource)','n_units':clust['n_units'],'lo':clust['lo'],'hi':clust['hi'],'width':clust['hi']-clust['lo']},
]).set_index('bootstrap').round(4))
print('\nThe clustered interval is the honest one. Report it.')

,n_units,lo,hi,width
bootstrap,,,,
naive (by instance),150,-0.0421,0.0307,0.0729
clustered (by resource),13,-0.0775,0.0500,0.1276



The clustered interval is the honest one. Report it.


## 4 · Base-model selection (Objective 2)



In [7]:
!pip -q install -U transformers accelerate bitsandbytes 2>&1 | tail -2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 110.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 19.5 MB/s eta 0:00:00


In [ ]:
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from mcasm.eval.metrics import evaluate_instance

EVAL_N = 60
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)

def eval_model(model_id, records, max_new_tokens=700):
    tk = AutoTokenizer.from_pretrained(model_id)
    if tk.pad_token is None: tk.pad_token = tk.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb, device_map='auto')
    mdl.eval()
    taus, valid, lat = [], 0, []
    for r in records:
        ids = [f['id'] for f in r['findings']]
        msgs = r['messages'][:2]
        prompt = tk.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        enc = tk(prompt, return_tensors='pt').to(mdl.device)
        t0 = time.time()
        with torch.no_grad():
            gen = mdl.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                               pad_token_id=tk.pad_token_id)
        lat.append(time.time()-t0)
        txt = tk.decode(gen[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
        parsed = parse_model_output(txt, ids)
        if parsed is None:
            taus.append(0.0); continue
        valid += 1
        taus.append(evaluate_instance(parsed['ranking'], r['target']['ranking'])['kendall_tau_b'])
    del mdl; torch.cuda.empty_cache()
    return {'model': model_id, 'tau_b': float(np.mean(taus)),
            'json_validity': valid/len(records), 'latency_s': float(np.mean(lat))}

sel = []
for mid in CANDIDATES:
    try:
        sel.append(eval_model(mid, splits['test'][:EVAL_N])); print(sel[-1])
    except Exception as e:
        print(f'{mid}: FAILED {type(e).__name__}: {e}')

if sel:
    seldf = pd.DataFrame(sel).sort_values('tau_b', ascending=False)
    display(seldf.round(4))
    ok = seldf[seldf['json_validity'] >= 0.95]
    BASE_MODEL = (ok if len(ok) else seldf).iloc[0]['model']
    print('\nSELECTED BASE MODEL:', BASE_MODEL)

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

## 5 · Export training-ready data


In [ ]:
from datetime import datetime, timezone
for sp, recs in splits.items():
    with open(OUT/f'{sp}.jsonl','w',encoding='utf-8') as fh:
        for r in recs:
            fh.write(json.dumps({'instance_id': r['instance_id'], 'split': sp,
                                 'messages': r['messages'],
                                 'input_order': [f['id'] for f in r['findings']],
                                 'target': r['compact_target']}, ensure_ascii=False)+'\n')
    print(f'{sp:11} {len(recs):5}')

man = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'format': 'compact-line',
    'max_seq_length': int(MAX_SEQ),
    'gpu': gpu, 'tier': TIER,
    'base_model': BASE_MODEL if 'BASE_MODEL' in dir() else None,
    'baselines': {k: v for k, v in results.items() if not k.endswith('_raw')},
    'model_selection': sel if sel else None,
    'note_on_B3': 'B3 shares a scoring function with the rule-derived gold ordering. Its score is a wiring check, not a result. The reportable comparison is against expert-annotated Corpus C in Notebook 06.',
    'note_on_ci': 'All CIs are clustered by resource. Instances are not independent (1500 instances drawn from 340 resources).',
}
(OUT/'manifest_03.json').write_text(json.dumps(man, indent=2, default=str))
print(json.dumps({k:v for k,v in man.items() if k!='baselines'}, indent=2, default=str))

In [ ]:
import shutil
arc = shutil.make_archive('/content/mcasm_train_ready','zip', OUT)
print('archive:', arc, round(Path(arc).stat().st_size/1024**2,2),'MiB')
from google.colab import files; files.download(arc)